# 02 — Compare YRBSS and NSDUH youth opioid misuse trends

In [ ]:
import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

In [ ]:
DATA_ROOT = Path('..') / 'data'
YR_DERIVED = DATA_ROOT / 'derived' / 'yrbss'
OUT_DIR = DATA_ROOT / 'derived' / 'yrbss_nsduh'
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load cleaned YRBSS series

In [ ]:
yr = pd.read_csv(YR_DERIVED / 'yrbss_opioid_trends.csv')
yr = yr.sort_values('year').copy()
yr.head()

## Scrape NSDUH detailed tables for adolescent pain reliever misuse

This mirrors the approach in `nsduh_opioid_prevalence/01_samhsa_tables.ipynb` by pulling annual SAMHSA detailed table ZIPs and extracting youth (12–17) prevalence entries.

In [ ]:
YEARS = list(range(2015, 2024))
BASE = 'https://www.samhsa.gov/data/sites/default/files/reports/rpt{{year}}/NSDUHDetailedTabs{{year}}/NSDUHDetailedTabs{{year}}.zip'

rows = []
for year in YEARS:
    url = BASE.format(year=year)
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f'Failed {year}: {e}')
        continue

    z = zipfile.ZipFile(io.BytesIO(r.content))
    html_names = [n for n in z.namelist() if re.search(r'sect1|sect5', n, flags=re.I)]
    for name in html_names:
        txt = z.read(name).decode('latin1', errors='ignore')
        soup = BeautifulSoup(txt, 'html.parser')
        table_text = soup.get_text(' ', strip=True)
        if ('12 to 17' in table_text or '12-17' in table_text) and ('pain reliever misuse' in table_text.lower() or 'prescription pain reliever misuse' in table_text.lower()):
            nums = re.findall(r'\d{1,2}\.\d', table_text)
            # first numeric in range 0-100 is a rough fallback when table parsing changes
            val = next((float(x) for x in nums if 0 <= float(x) <= 100), None)
            if val is not None:
                rows.append({'year': year, 'prevalence_pct': val, 'indicator': 'Past-year prescription pain reliever misuse (12-17)', 'source': 'NSDUH'})
                break

ns = pd.DataFrame(rows).drop_duplicates(subset=['year'])
ns

if ns.empty:
    print('No NSDUH rows parsed from SAMHSA ZIP files. You may need to update parser patterns or run notebook 01 in nsduh_opioid_prevalence to build a reference extract.')


> If scraping misses years due to table format changes, manually inspect notebook `nsduh_opioid_prevalence/01_samhsa_tables.ipynb` and update the parser pattern accordingly.

In [ ]:
# Harmonize and join
yr2 = yr[['year','prevalence_pct','ci_low','ci_high','source']].copy()
yr2['series'] = 'YRBSS: Rx pain medicine misuse'

if ns.empty:
    ns2 = pd.DataFrame(columns=['year','prevalence_pct','source','ci_low','ci_high','series'])
else:
    ns2 = ns[['year','prevalence_pct','source']].copy()
    ns2['ci_low'] = np.nan
    ns2['ci_high'] = np.nan
    ns2['series'] = 'NSDUH: Past-year Rx misuse (12-17)'

combo = pd.concat([yr2, ns2], ignore_index=True).sort_values(['series','year'])
combo.to_csv(OUT_DIR / 'yrbss_nsduh_opioid_trends.csv', index=False)
combo.tail()

## Trend visualization with uncertainty

Per project convention, line plots include uncertainty bounds. For YRBSS we use published CIs; for NSDUH we show points/line only unless CIs are added from parsed tables.

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

for series, d in combo.groupby('series'):
    d = d.sort_values('year')
    if d['ci_low'].notna().any() and d['ci_high'].notna().any():
        yerr = [d['prevalence_pct'] - d['ci_low'], d['ci_high'] - d['prevalence_pct']]
        ax.errorbar(d['year'], d['prevalence_pct'], yerr=yerr, capsize=3, marker='o', label=series)
    else:
        ax.plot(d['year'], d['prevalence_pct'], marker='o', label=series)

ax.set_title('Youth opioid misuse trends: YRBSS vs NSDUH')
ax.set_xlabel('Survey year')
ax.set_ylabel('Prevalence (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Summary metrics
summary = (combo.groupby('series')
    .apply(lambda g: pd.Series({
        'first_year': int(g['year'].min()),
        'last_year': int(g['year'].max()),
        'first_prevalence': float(g.sort_values('year')['prevalence_pct'].iloc[0]),
        'last_prevalence': float(g.sort_values('year')['prevalence_pct'].iloc[-1]),
        'absolute_change_pct_points': float(g.sort_values('year')['prevalence_pct'].iloc[-1] - g.sort_values('year')['prevalence_pct'].iloc[0]),
        'relative_change_pct': float((g.sort_values('year')['prevalence_pct'].iloc[-1] / g.sort_values('year')['prevalence_pct'].iloc[0] - 1) * 100),
    }))
    .reset_index())

summary.to_csv(OUT_DIR / 'summary_metrics.csv', index=False)
summary